# Apply the Llama model 70B online

## Information

Inference API of Hugging Face exposes models that have large community interest and are in active use: https://huggingface.co/docs/api-inference/supported-models


**Precondition**: create an Access Token (https://huggingface.co/settings/tokens), set up a pro account to use the larger LLMs like Llama-3-70B (https://huggingface.co/pricing#pro) and accept the META LLAMA 3 COMMUNITY LICENSE AGREEMENT for the two different Llama models:

* for `meta-llama/Meta-Llama-3-70B-Instruct`: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct

Remark: Llama models are published under the **META LLAMA 3 COMMUNITY LICENSE AGREEMENT**. The Meta Llama 3 Community License grants users a non-exclusive, royalty-free license (you not need to pay ongoing fees) to use, modify, and distribute Llama 3 materials, with requirements for attribution and naming conventions when creating derivative works. Users with over 700 million monthly active users need a separate license, and Meta disclaims all warranties and limits liability for any use of the materials.


***
**Coding sources**


* You can run the `meta-llama/Meta-Llama-3-70B-Instruct`, see model page: https://huggingface.co/meta-llama/Meta-Llama-3-70B-Instruct
    + Hugging Face documentation: https://huggingface.co/docs/transformers/main/en/model_doc/llama3

## Load necessary libraries and data:

In [1]:
import os
import sys

# Assuming 'src' is two levels down (in the current directory or a subdirectory)
path_to_src = os.path.join('..')  # Moves one level down to 'src' folder

# Change the working directory
os.chdir(path_to_src)

# Verify the updated working directory
print("Updated working directory:", os.getcwd())

# load helper functions
import src.prompt_functions as pf
import src.prompt_text as pt
import src.get_data as get_data
import src.API_key as keys

Updated working directory: /home/fenn/Desktop/Publications/basal attributes/Analyses/part_III


load external data:

words of the single partitions and their respective mean_valence:

In [2]:
import pandas as pd

df_partitions = pd.read_excel('./data/LeidenAlgorithm_solution.xlsx')
df_partitions.rename(columns={'parition': 'partition'}, inplace=True)

In [3]:
df_partitions

,partition,words,mean_valence,sd_valence
0,1,aktive Formänderung durch Umwelteinwirkung,0.18,1.140892
1,1,reaktionsfähig,0.67,1.111209
2,1,passive Formänderung durch Umwelteinwirkung,-0.02,1.131126
3,1,autonom,0.75,0.949819
4,1,passive Verhaltensänderung durch Umwelteinwirkung,0.06,1.056184
5,1,aktive Verhaltensänderung durch Umwelteinwirkung,0.27,1.148185
6,1,intelligent,1.35,1.103033
7,1,multifunktional,1.65,1.024702
8,1,technologisch,0.76,0.959081
9,2,zuverlässig,2.17,1.017770


combination of partitions I want to generate a text from:

In [4]:
df_hypothesis= pd.read_excel('./data/LeidenAlgorithm_hypotheses.xlsx')
df_hypothesis.rename(columns={'partitionA': 'partitionA', 'partitionB': 'partitionB'}, inplace=True)


In [5]:
df_hypothesis = df_hypothesis[df_hypothesis['choose'] == 1][['partitionA', 'partitionB']]
df_hypothesis = df_hypothesis.reset_index(drop=True)
df_hypothesis

,partitionA,partitionB
0,3,6
1,3,5
2,1,6
3,1,5
4,1,2
5,2,4


In [6]:
import pandas as pd

# Assuming df_partitions and df_hypothesis are already defined

# Function to compute mean and std for combined partitions
def compute_partition_stats(row):
    partition_a = row['partitionA']
    partition_b = row['partitionB']
    
    # Filter df_partitions for the two partitions
    subset = df_partitions[df_partitions['partition'].isin([partition_a, partition_b])]
    
    # Compute mean and std of mean_valence
    mean_val = subset['mean_valence'].mean()
    std_val = subset['mean_valence'].std()
    
    return pd.Series({'computed_mean': mean_val, 'computed_sd': std_val})

# Apply function to each row in df_hypothesis
df_hypothesis[['computed_mean', 'computed_sd']] = df_hypothesis.apply(compute_partition_stats, axis=1)

df_hypothesis


,partitionA,partitionB,computed_mean,computed_sd
0,3,6,-0.927143,1.288264
1,3,5,-0.093750,1.954942
2,1,6,0.555000,0.545952
3,1,5,0.953846,0.706830
4,1,2,1.220000,0.775879
5,2,4,1.858333,0.516682


## Load the prompt template and model:

In [7]:
from importlib import reload

pt = reload(pt)
prompt_text = pt.template_partitions
prompt_text

'\n\n\n\n<Aufgabe:\nEntwickle eine für Laien verständliche Beschreibung eines neuen Materialsystems für den Einsatz in einem realen Bau- oder Infrastrukturprojekt. Das Projekt sollte die Eigenschaften des Materials nutzen, um bestehende Probleme effektiv zu adressieren, während es auch auf mögliche Herausforderungen und Schwierigkeiten des Materialsystems eingeht. \n\n\nGenrell zeichnet sich das neue Materialsystem durch neuartige Eigenschaften aus, die es von herkömmlichen Materialien unterscheidet. Verwende hierzu alle Wörter aus der folgenden Liste, um die neuartigen Eigenschaften, aber auch die bestehenden Herausforderungen des Materialsystems zu beschreiben.>\n\n<Liste:\n({items_list})>\n\n<Aufgabenstellungen:\n1) Der Text muss weniger als 150 Wörter beinhalten.\n2) Schreibe neutral aus der Dritten Person und beziehe dich auf das konkrete Projekt.\n3) Achte bei der Verwendung der Wörter darauf, nicht Wörter mit unterschiedlichen emotionalen Konnotationen zu vermischen.\n4) Das Pro

aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [8]:
from langchain_openai import ChatOpenAI
from langchain.callbacks import get_openai_callback

def huggingface_API_call(
    prompt,
    
    items_list,

    api_key=keys.hugging_api_key,

    model_name="meta-llama/llama-3.3-70b-instruct",
    json_schema=None,
    max_tokens=1000,
    temperature=0.0,
    verbose=True
):

    # Initialize the ChatOpenAI model for Hugging Face
    model = ChatOpenAI(
        model=model_name,
        openai_api_key=api_key,
        openai_api_base="https://router.huggingface.co/novita/v3/openai",
        max_tokens=max_tokens,
        temperature=temperature
    )

    # Check if structured output is required and configure it
    if json_schema:
        structured_llm = model.with_structured_output(json_schema, include_raw=True)
        chain = prompt | structured_llm
    else:
        chain = prompt | model

    # Execute the model and output response details
    with get_openai_callback() as cb:
        response = chain.invoke(
            {"items_list": items_list}
        )
        
        if cb.total_tokens > max_tokens:
            print("Warning: The response may be incomplete due to exceeding the maximum token limit.")
        
        if verbose:
            print(cb)
            print(f"Total Tokens: {cb.total_tokens}")
            print(f"Prompt Tokens: {cb.prompt_tokens}")
            print(f"Completion Tokens: {cb.completion_tokens}")
            print(f"Total Cost (USD): ${cb.total_cost}")

    return response

aaaaaaaaaaaaaa

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
<Aufgabe:
Entwickle eine für Laien verständliche, aber wissenschaftlich fundierte Beschreibung eines neuartigen Jackensystems namens Nano-Pat-Parka. 
Ziel ist es, anhand sogenannter basaler Attribute – also grundlegender, semantisch und emotional bewerteter Merkmale neuer Technologien – die möglichen Vorteile und Herausforderungen dieser Technologie zu verdeutlichen.

Basale Attribute sind adjektivische Eigenschaften (z. B. „autonom“, „wartungsintensiv“, „bioinspiriert“), die typischerweise verwendet werden, um neue technologische Systeme zu beschreiben. 
Sie dienen der kognitiven und affektiven Bewertung und ermöglichen eine strukturierte Beschreibung technischer Systeme auf einer allgemeinen Ebene – unabhängig von konkreten Details.

Die Beschreibung des Nano-Pat-Parka soll sich mehrheitliich auf diese basalen Attribute stützen, ohne große Ausführungen zu konkreten technischen Funktionen oder mechanistischen Erklärungen. 
Das Jackensystem soll funktional generisch beschrieben werden – als Technologie, die auf zukünftige Anforderungen im Bereich Schutzkleidung reagiert.>

<Aufgabenstellungen:
1) ***Der Text MUSS zwischen 60 und 80 Wörter enthalten.*** Weniger als 60 Wörter sind nicht zulässig. Kontrolliere dieses bevor du den Text absendest.
2) Schreibe sachlich, neutral und aus der dritten Person.
3) Achte darauf, keine basalen Attribute mit stark gegensätzlicher emotionaler Valenz zu vermischen.
4) Die Beschreibung muss wissenschaftlich plausibel und begrifflich konsistent sein.
5) Beginne mit einem einleitenden Satz, der den Anwendungsbereich der Technologie allgemein beschreibt.
6) Verwende ALLE Wörter, die in der Liste angegeben sind.
7) Schreibe einen abschließenden Satz, der die Technologie zusammenfasst.>
"""

user_template = """
<Liste der zu verwendenden basalen Attribute: 
({items_list})>
"""

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", user_template)]
)



In [ ]:
for index, row in df_hypothesis.iterrows():
    print(f"Index {index}")

    # Ensure the value is list-like
    partitionA = [row['partitionA']]
    partitionB = [row['partitionB']]
    combined_partitions = list(partitionA) + list(partitionB)

    combined_subset = df_partitions[df_partitions['partition'].isin(combined_partitions)]["words"].values
    combined_subset = " // ".join(combined_subset.tolist())

Index 0
Index 1
Index 2
Index 3
Index 4
Index 5


In [43]:
combined_subset

'wartungsintensiv // enthält Kunststoff // leicht zerstörbar // umweltschädlich // Insekten ähnlich // bioinspiriert // lebensähnlich'

In [44]:
result = huggingface_API_call(prompt=prompt_template,
                     items_list=combined_subset,
                     model_name="deepseek/deepseek-r1-turbo", temperature=0)

Tokens Used: 1015
	Prompt Tokens: 498
	Completion Tokens: 517
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 1015
Prompt Tokens: 498
Completion Tokens: 517
Total Cost (USD): $0.0


In [38]:
result = huggingface_API_call(prompt=prompt_template,
                     items_list=combined_subset,
                     model_name="meta-llama/llama-3.3-70b-instruct", temperature=0)

Tokens Used: 674
	Prompt Tokens: 572
	Completion Tokens: 102
Successful Requests: 1
Total Cost (USD): $0.0
Total Tokens: 674
Prompt Tokens: 572
Completion Tokens: 102
Total Cost (USD): $0.0


In [46]:
result

AIMessage(content='<think>\nOkay, let\'s tackle this. The user wants a description of the Nano-Pat-Parka using specific attributes. First, I need to make sure I understand all the requirements. The text must be 60-80 words, neutral, third person, and use all the listed attributes. The attributes given are: wartungsintensiv, enthält Kunststoff, leicht zerstörbar, umweltschädlich, Insekten ähnlich, bioinspiriert, lebensähnlich. \n\nStarting with the introductory sentence about the application in protective clothing. Then, incorporating each attribute. "Bioinspiriert" and "Insekten ähnlich" can be linked, maybe the design mimics insect exoskeletons. "Lebensähnlich" suggests adaptive or responsive features. Now the challenges: "wartungsintensiv" and "leicht zerstörbar" indicate maintenance and fragility. "Enthält Kunststoff" and "umweltschädlich" relate to environmental impact. Need to present these neutrally, not mixing opposing emotions.\n\nCheck if all attributes are used. Let me count:

In [45]:
print(result.content)

word_count = len(result.content.split())
print("Number of words:", word_count)

<think>
Okay, let's tackle this. The user wants a description of the Nano-Pat-Parka using specific attributes. First, I need to make sure I understand all the requirements. The text must be 60-80 words, neutral, third person, and use all the listed attributes. The attributes given are: wartungsintensiv, enthält Kunststoff, leicht zerstörbar, umweltschädlich, Insekten ähnlich, bioinspiriert, lebensähnlich. 

Starting with the introductory sentence about the application in protective clothing. Then, incorporating each attribute. "Bioinspiriert" and "Insekten ähnlich" can be linked, maybe the design mimics insect exoskeletons. "Lebensähnlich" suggests adaptive or responsive features. Now the challenges: "wartungsintensiv" and "leicht zerstörbar" indicate maintenance and fragility. "Enthält Kunststoff" and "umweltschädlich" relate to environmental impact. Need to present these neutrally, not mixing opposing emotions.

Check if all attributes are used. Let me count: bioinspiriert, Insekten 

In [41]:
print(result.content)

word_count = len(result.content.split())
print("Number of words:", word_count)

<think>
Okay, let me start by understanding the user's request. They want a description of a new jacket system called Nano-Pat-Parka, focusing on specific attributes provided. The text needs to be between 60-80 words, in German, neutral and third-person. The challenge is to use all the listed adjectives without mixing conflicting emotional valences.

First, I'll list the attributes: zuverlässig, wartungsfrei, selbstheilend, widerstandsfähig, selbstreparierend, haltbar, robust, langlebig, ökologisch, elektronikfrei, nachhaltig, umweltfreundlich. I need to ensure all these are included. Some are similar, like selbstheilend and selbstreparierend, so I should use them in a way that makes sense without redundancy.

Next, the structure. The introduction should mention the application area, which is protective clothing for future demands. Then, each sentence should incorporate a few attributes. I need to avoid technical details, keeping it generic. The conclusion should summarize the technolo

In [ ]:
pi=reload(pf)
chosen_partitions = [1, 2] 
examplary_template = pf.prompt_template(df_partitions, chosen_partitions, prompt_text)
print(examplary_template)

In [ ]:
key=reload(key)
api_key = key.openai_api_key

In [ ]:
## remove tuples
import itertools
unique_partitions = df_partitions["partition"].unique()
all_combinations = list(itertools.combinations(unique_partitions, 2))
print(all_combinations)
# Tuples to remove
to_remove = {(3, 5), (2, 3)}
# Filter out the tuples that are to be removed
filtered_combinations = [t for t in all_combinations if t not in to_remove]
print(filtered_combinations)



In [ ]:
result, full_response = pi.process_partitions(df_partitions, prompt_text, api_key, model="gpt-4", chosen_partitions=[(3,6)], max_tokens=500, max_API_calls=1) 

In [ ]:
full_response

In [ ]:
result

In [ ]:
get_data.save_results_to_csv('result.csv', result)

In [ ]:
result_df = pd.read_csv('./output/result.csv')

In [ ]:
get_data=reload(get_data)
text_check = get_data.check_for_missing_matching_words(result_df['text'], df_partitions, [3,6], 0.5)
print(text_check)